# LC 295 — Find Median from Data Stream
**Difficulty:** Hard &nbsp;|&nbsp; **Category:** Heap
**Pattern:** Two Heaps — Max-Heap Lower Half +
Min-Heap Upper Half

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Split all numbers
into a lower half (max-heap) and upper half
(min-heap), kept balanced in size. The median is
the top of the larger heap, or the average of
both tops when sizes are equal.
</div>

## Official Problem Statement

The **median** is the middle value in an ordered
integer list. If the size of the list is even,
there is no middle value, and the median is the
mean of the two middle values.

Implement the `MedianFinder` class:

- `MedianFinder()` Initialises the object.
- `void addNum(int num)` Adds the integer `num`
  from the data stream to the data structure.
- `double findMedian()` Returns the median of all
  elements so far. Answers within `10^-5` of the
  actual answer will be accepted.

**Example 1:**
```
Input:
  ["MedianFinder","addNum","addNum","findMedian",
   "addNum","findMedian"]
  [[],[1],[2],[],[3],[]]
Output:
  [null, null, null, 1.5, null, 2.0]
```

**Constraints:**
- `-10^5 <= num <= 10^5`
- At most `5 * 10^4` calls to `addNum` and
  `findMedian`
- There will be at least one element before
  `findMedian` is called

## What This Is Actually Asking

Numbers arrive one at a time in any order.
After each addition you may be asked: if all
numbers so far were sorted, what is the middle
value? If there are two middle values, return
their average. Do this without re-sorting every
time.

## Walk Through an Example by Hand

```
addNum(1):
  Push 1 to lo (max-heap, store as -1)
  lo=[-1]  hi=[]
  sizes: lo=1 hi=0 — balanced

addNum(2):
  Push 2 to lo -> lo=[-2,-1]
  lo now larger by 2 -> rebalance:
    pop lo max=2, push to hi -> hi=[2]  lo=[-1]
  sizes: lo=1 hi=1 — balanced

findMedian():
  sizes equal -> (lo_top + hi_top) / 2
  = (-(-1) + 2) / 2 = 1.5

addNum(3):
  Push 3 to lo -> lo=[-3,-1]
  3 > hi top (2) -> wrong heap!
  Better strategy: always push to lo, then
  if lo top > hi top move lo top to hi,
  then rebalance sizes.

  push -3 to lo -> lo=[-3,-1] hi=[2]
  lo top=3 > hi top=2 -> pop 3 from lo, push to hi
  lo=[-1]  hi=[2,3]
  hi has 2, lo has 1 -> pop hi min=2, push to lo
  lo=[-2,-1]  hi=[3]

findMedian():
  lo has more -> return -lo[0] = 2.0
```

## The Picture

```
All numbers sorted: [1, 2, 3, 4, 5, 6, 7]
                         |
                      median

Split at the middle:

  lo (max-heap)   |  hi (min-heap)
  [1, 2, 3]       |  [4, 5, 6, 7]
         ^              ^
         3              4   <- the two midpoints

  lo top = 3 (largest of lower half)
  hi top = 4 (smallest of upper half)

Balance rule:
  sizes may differ by at most 1.
  If lo has more: median = lo top
  If hi has more: median = hi top
  If equal:       median = (lo top + hi top) / 2

Insert rule for addNum(x):
  1. push x to lo (max-heap, negate)
  2. if lo top > hi top: move lo top to hi
     (keep lo <= hi constraint)
  3. if |lo| > |hi| + 1: move lo top to hi
     if |hi| > |lo|:     move hi top to lo
     (keep sizes balanced)
```

## When To Use This Pattern

- When you need **running median from a stream**,
  think **two heaps — max-heap lower, min-heap upper**
- When inserting, think **always push to lo first,
  then fix ordering and balance**
- When sizes are equal, think
  **average of both tops**
- When one heap is larger by 1, think
  **median is the top of the larger heap**
- When lo top > hi top, think
  **rebalance: move lo top to hi**

## The Approach

Maintain two heaps: lo is a max-heap (negated
values) for the lower half, hi is a min-heap for
the upper half.
On every addNum: push to lo, then if lo's top
exceeds hi's top move lo's top to hi. Then rebalance
sizes so neither heap exceeds the other by more
than one.
findMedian returns the average of both tops when
sizes are equal, otherwise the top of the larger
heap.

In [ ]:
import heapq  # lo: negate values for max-heap; hi: normal min

In [ ]:
def test_harness(cls):
    """
    Replay sequences of (op, args, expected) against cls.
    ops: 'add'  -> (num,)   expected ignored (None)
         'find' -> ()       expected is the median float
    """
    sequences = [
        # sequence 1 — problem example
        [
            ("add",  (1,),  None),
            ("add",  (2,),  None),
            ("find", (),    1.5),
            ("add",  (3,),  None),
            ("find", (),    2.0),
        ],
        # sequence 2 — single element
        [
            ("add",  (5,),  None),
            ("find", (),    5.0),
        ],
        # sequence 3 — odd then even count
        [
            ("add",  (6,),  None),
            ("add",  (2,),  None),
            ("add",  (4,),  None),
            ("find", (),    4.0),  # sorted:[2,4,6] median=4
            ("add",  (8,),  None),
            ("find", (),    5.0),  # sorted:[2,4,6,8] median=5
        ],
        # sequence 4 — duplicates
        [
            ("add",  (3,),  None),
            ("add",  (3,),  None),
            ("find", (),    3.0),
        ],
        # sequence 5 — negatives
        [
            ("add",  (-1,), None),
            ("add",  (-2,), None),
            ("find", (),   -1.5),
            ("add",  (-3,), None),
            ("find", (),   -2.0),
        ],
    ]

    passed = 0
    for s_i, seq in enumerate(sequences):
        obj = cls()
        seq_pass = True
        for op, args, expected in seq:
            if op == "add":
                obj.addNum(*args)
            else:
                result = obj.findMedian()
                ok = abs(result - expected) < 1e-5
                if not ok:
                    seq_pass = False
                    print(
                        f"  Seq {s_i+1} FAILED: "
                        f"findMedian expected "
                        f"{expected} got {result}"
                    )
        if seq_pass:
            passed += 1
            print(f"Sequence {s_i+1}: PASSED")
        else:
            print(f"Sequence {s_i+1}: FAILED")

    print(f"\n{passed}/{len(sequences)} sequences passed")

In [ ]:
class MedianFinder:
    """
    Running median from a data stream using two heaps.

    lo: max-heap (negated) for lower half.
    hi: min-heap for upper half.
    addNum: push to lo, fix order (lo top <= hi top),
    then rebalance sizes (differ by at most 1).
    findMedian: average tops if equal size, else top
    of larger heap.

    addNum:    O(log n) — two heap operations
    findMedian: O(1)   — peek both tops
    Space: O(n)
    """

    def __init__(self):
        pass

    def addNum(self, num: int) -> None:
        pass

    def findMedian(self) -> float:
        pass


# Quick debug — run this cell while building
mf = MedianFinder()
mf.addNum(1)
mf.addNum(2)
print(mf.findMedian())  # 1.5
mf.addNum(3)
print(mf.findMedian())  # 2.0
mf.addNum(4)
print(mf.findMedian())  # 2.5

In [ ]:
# Uncomment and run when solution is ready
# test_harness(MedianFinder)

## Complexity

| Approach | addNum | findMedian | Space |
|---|---|---|---|
| Sort on every query | O(n log n) | O(1) | O(n) |
| Sorted list insert | O(n) | O(1) | O(n) |
| Two heaps | O(log n) | O(1) | O(n) |

Two heaps is optimal: each add touches at most
three heap operations (push + push + pop) all
O(log n), and the median is always a single
O(1) peek.

## Real World Connection

At Citi, the Prophet forecasting pipeline needs a
rolling median CPU utilisation per server to
filter outlier spikes before training the model.
Readings arrive in a stream; re-sorting 90 days
of data after every new reading is too slow.
The two-heap MedianFinder inserts each new reading
in O(log n) and returns the median in O(1), keeping
the preprocessing stage inside the pipeline's
latency budget.
On AWS Kinesis Data Streams, the same pattern runs
as a sliding-window aggregator: the two heaps
rebalance on each record arrival and emit a median
metric downstream without buffering the full window.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra